# RF Board — ADC Test Notebook

Tests the AD9643 ADC at 100 MHz sample rate.

**Sequence:**
1. Load PL bitstream overlay
2. Instantiate `PynqController`
3. Read ADC chip identification
4. Configure CDCE62005 — DAC 1 GHz, ADC 100 MHz
5. Initialize DACs
6. Initialize ADC
7. ADC register readback and test modes

In [2]:
from pynq import Overlay
import time
from pynq_controller import PynqController
ol = Overlay('./ADC_DAC12ST_DATACLK125MHz.xsa')

print('Overlay loaded.')

Overlay loaded.


In [3]:
DDS_CLK_HZ= 62_500_000
BOARD_VER = 2
zynq = PynqController(ol, board_ver=BOARD_VER, enable_bram=False, enable_axi=True)

zynq.adc_init()

# Supply DACCLK and OSTR: DACCLK1 DACCLK2 1 GHz, OSTR effective 15.625 MHz, FPGA_CLK ADC_CLK off
CDCE_CONFIG_no_FPGA = '8340032083400321EA040302811C0303EA84031410000BE504BE09F6BD0037F7'
zynq.configure_clock(CDCE_CONFIG_no_FPGA,debug=1)



[PynqController] Initializing (board v2)...
CDCE62005 ready (board v2): CLK=MIO12, MOSI=MIO10, MISO=MIO11, LE=MIO15
  CS_DAC1=MIO13, CS_DAC2=MIO14, CS_ADC=MIO0 — all deasserted
SPI3WireBus ready (board v2): CLK=MIO12, SDIO=MIO11
DAC3484 DAC1 ready (CS_DAC1)
DAC3484 DAC2 ready (CS_DAC2)
AD9643 ADC ready
  (BRAM controller disabled)
    ⚠ DDS_CTRL GPIO not found in overlay (ST_0/DDS_CTRL). Verify XSA has DDS_CTRL AXI GPIO block.
  (AXI subsystem initialized for DDS)
✓ PynqController ready


[ADC] Initializing AD9643...
  AD9643 Chip ID: 0x82 (expected 0x82) [OK]
✓ ADC initialized

Configuring CDCE62005...
  reg[0] <- 0x8340032
  reg[1] <- 0x8340032
  reg[2] <- 0xEA04030
  reg[3] <- 0x811C030
  reg[4] <- 0xEA84031
  reg[5] <- 0x10000BE
  reg[6] <- 0x04BE09F
  reg[7] <- 0xBD0037F
  reg[8] = 0x20009FF (OK)
Done.


True

In [4]:
# ___________________________________________________________________________________________________________________
#--------------------------------------------- DAC2 Initialization- Part 1 ------------------------------------------
### **Program SIF registers:**
zynq.dac2.write_register(0x00, 0x089F) # INTERP = 16, fifo on, clkdiv_sync_ena invsincAB+CD
zynq.dac2.write_register(0x01, 0x050E)
zynq.dac2.write_register(0x02, 0xF052) # 16bit_in, mixer_ena, nco_ena, twos_comp
zynq.dac2.write_register(0x03, 0xF000)
zynq.dac2.write_register(0x07, 0xD8FF) # un-mask FIFO collison, DACCLK-gone, DATACLK-gone

# # registers 0x08-0x11 are QMC module, unused
# # registers 0x12-0x13 NCO phase, unused 

# set_dac_nco(self, channel: int, freq_mhz: float, phase_deg: float = 0)
zynq.dac2.set_dac_nco(1, 00.0, 0)
zynq.dac2.set_dac_nco(2, 00.0, 0)

# PLL Disable:
zynq.dac2.write_register(0x18, 0x0000) # pll_ena=0, 
zynq.dac2.write_register(0x19, 0x0000)
zynq.dac2.write_register(0x1A, 0x0020) # pll_sleep=1

# # PLL Enable:
# zynq.dac2.write_register(0x18, 0x0460)  # pll_ena=1, pll_cp[1:0]=01 single charge pump, pll_p[2:0]=100 (4)
# zynq.dac2.write_register(0x19, 0x0434)  # pll_m[7:0]= 0 000 0100 (4), pll_n[3:0]=0011 (4), pll_vcoitune[1:0]=01
# zynq.dac2.write_register(0x1A, 0xFC00)  # pll_vco[5:0]=1111 11 (4GHz), pll_sleep=0
# lfvolt2 = zynq.dac2.read_register(0x18) # read pll_lfvolt to check for lock
# print(f"DAC2- PLL LOCK : REG0x18 = 0x{lfvolt2:04X}")

zynq.dac2.write_register(0x1B, 0x0800) # fuse_sleep=1

zynq.dac2.write_register(0x1E, 0x8888) # sync_sel for QMC module- SIF_SYNC
zynq.dac2.write_register(0x1F, 0x2224) # syncsel_mixerAB, syncsel_mixerCD, syncsel_nco: OSTR, syncsel_dataformatter: SYNC 
zynq.dac2.write_register(0x20, 0x1400) # syncsel_fifoin: SYNC, syncsel_fifoout: OSTR, clkdiv_sync_sel: OSTR
# ___________________________________________________________________________________________________________________


# ___________________________________________________________________________________________________________________
# xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx DAC1 Initialization- Part 1 xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
zynq.dac1.write_register(0x00, 0x089F) # INTERP = 16, fifo on, clkdiv_sync_ena invsincAB+CD
zynq.dac1.write_register(0x01, 0x050E)
zynq.dac1.write_register(0x02, 0xF052) # 16bit_in, mixer_ena, nco_ena, twos_comp
zynq.dac1.write_register(0x03, 0xF000)
zynq.dac1.write_register(0x07, 0xD8FF) # un-mask FIFO collison, DACCLK-gone, DATACLK-gone
time.sleep(0.1)

# # registers 0x08-0x11 are QMC module, unused
# # registers 0x12-0x13 NCO phase, unused 

zynq.dac1.set_dac_nco(1, 00.0, 0)
zynq.dac1.set_dac_nco(2, 00.0, 0)

# PLL Disable:
zynq.dac1.write_register(0x18, 0x0000) # pll_ena=0, 
zynq.dac1.write_register(0x19, 0x0000)
zynq.dac1.write_register(0x1A, 0x0020) # pll_sleep=1

# # PLL Enable:
# zynq.dac1.write_register(0x18, 0x0460)  # pll_ena=1, pll_cp[1:0]=01 single charge pump, pll_p[2:0]=100 (4)
# zynq.dac1.write_register(0x19, 0x0434)  # pll_m[7:0]= 0 000 0100 (4), pll_n[3:0]=0011 (4), pll_vcoitune[1:0]=01
# zynq.dac1.write_register(0x1A, 0xFC00)  # pll_vco[5:0]=1111 11 (4GHz), pll_sleep=0
# lfvolt2 = zynq.dac1.read_register(0x18) # read pll_lfvolt to check for lock
# print(f"DAC1- PLL LOCK: REG0x18 = 0x{lfvolt2:04X}")

zynq.dac1.write_register(0x1B, 0x0800) # fuse_sleep=1

zynq.dac1.write_register(0x1E, 0x8888) # sync_sel for QMC module- SIF_SYNC
zynq.dac1.write_register(0x1F, 0x2224) # syncsel_mixerAB, syncsel_mixerCD, syncsel_nco: OSTR, syncsel_dataformatter: SYNC 
zynq.dac1.write_register(0x20, 0x1400) # syncsel_fifoin: SYNC, syncsel_fifoout: OSTR, clkdiv_sync_sel: OSTR
# ___________________________________________________________________________________________________________________


# ___________________________________________________________________________________________________________________
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Clock configuration for both DACs ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
## Add FPGACLK in order to start all LVDS data and clocks
# DACCLK1 DACCLK2 1 GHz, OSTR effective 15.625 MHz, FPGA_CLK 250 MHz, ADC_CLK off
CDCE_CONFIG_wFPGA = '8340032083400321EB840302811C0303EB84031410000BE504BE09F6BD0037F7'
zynq.configure_clock(CDCE_CONFIG_wFPGA)
print('CDCE configured with FPGACLK')
time.sleep(0.001)
# ___________________________________________________________________________________________________________________


# ___________________________________________________________________________________________________________________
#--------------------------------------------- DAC2 Initialization- Part 2 -------------------------------------------
## Check pll lock and reset alarms register
lfvolt2 = zynq.dac2.read_register(0x18) # read pll_lfvolt to check for lock
print(f"✓ PLL LOCK: REG0x18 = 0x{lfvolt2:04X}")

zynq.dac2.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC2 = zynq.dac2.read_register(0x05) # read reg5 to check for alarms
print(f"DAC2- Alarms Register: REG0x05 = 0x{reg5_DAC2:04X}")
time.sleep(0.001)

### Disable clock divider sync
zynq.dac2.write_register(0x1F, 0x2224) # turn on sif_sync
zynq.dac2.write_register(0x00, 0x089B) # INTERP = 16, fifo on, clkdiv_sync_ena=0, invsincAB+CD=1
time.sleep(0.001)
zynq.dac2.write_register(0x1F, 0x2228) # turn off sif_sync, syncsel_dataformatter- no sync
zynq.dac2.write_register(0x20, 0x0000) # Disable FIFO input and output pointer sync
zynq.dac2.write_register(0x03, 0xF001) # Set TXENABLE high
zynq.dac2.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC2 = zynq.dac2.read_register(0x05) # read reg5 to check for alarms
print(f"DAC2- Alarms Register : REG0x05 = 0x{reg5_DAC2:04X}")
zynq.dac2.write_register(0x24, 0x1C00)  
# ___________________________________________________________________________________________________________________


# ___________________________________________________________________________________________________________________
# xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx DAC1 Initialization- Part 2 xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
## Check pll lock and reset alarms register
lfvolt1 = zynq.dac1.read_register(0x18) # read pll_lfvolt to check for lock
print(f"DAC1- PLL LOCK: REG0x18 = 0x{lfvolt1:04X}")

zynq.dac1.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC1 = zynq.dac1.read_register(0x05) # read reg5 to check for alarms
print(f"DAC1- Alarms Register: REG0x05 = 0x{reg5_DAC1:04X}")
time.sleep(0.001)

### Disable clock divider sync
zynq.dac1.write_register(0x1F, 0x2224) # turn on sif_sync
zynq.dac1.write_register(0x00, 0x089B) # INTERP = 16, fifo on, clkdiv_sync_ena=0, invsincAB+CD=1
time.sleep(0.001)
zynq.dac1.write_register(0x1F, 0x2228) # turn off sif_sync, syncsel_dataformatter- no sync


zynq.dac1.write_register(0x20, 0x0000) # Disable FIFO input and output pointer sync
zynq.dac1.write_register(0x03, 0xF001) # Set TXENABLE high
zynq.dac1.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC1 = zynq.dac1.read_register(0x05) # read reg5 to check for alarms
print(f"DAC1- Alarms Register: REG0x05 = 0x{reg5_DAC1:04X}")
zynq.dac1.write_register(0x24, 0x1C00) 
# ___________________________________________________________________________________________________________________

DAC2 NCO configured: word=0x00000000
DAC2 NCO configured: word=0x00000000
DAC1 NCO configured: word=0x00000000
DAC1 NCO configured: word=0x00000000
CDCE configured with FPGACLK
✓ PLL LOCK: REG0x18 = 0x0007
DAC2- Alarms Register: REG0x05 = 0x3978
DAC2- Alarms Register : REG0x05 = 0x1878
DAC1- PLL LOCK: REG0x18 = 0x0007
DAC1- Alarms Register: REG0x05 = 0x3978
DAC1- Alarms Register: REG0x05 = 0x1878


In [5]:
# Step 5b — Configure DAC2 DDS (PL fabric) for test signal generation
from pynq_dds import DAC2_DDS

dds2 = DAC2_DDS(ol, dds_clk_hz=DDS_CLK_HZ)

# Configurable frequency and phase
AB_FREQ = 5.0    # MHz
AB_PHASE = 0.0   # degrees
CD_FREQ = 5.0    # MHz
CD_PHASE = 0.0   # degrees

dds2.set(ab_freq_mhz=AB_FREQ,  ab_phase_deg=AB_PHASE,
         cd_freq_mhz=CD_FREQ,  cd_phase_deg=CD_PHASE)

print(f"\n✓ DAC2 DDS configured ({DDS_CLK_HZ/1e6:.1f} MHz clock):")
print(f"  AB: {AB_FREQ} MHz @ {AB_PHASE}°")
print(f"  CD: {CD_FREQ} MHz @ {CD_PHASE}°")

DAC2 DDS ready  (CLK = 62.5 MHz)
AB: 5.000 MHz  0.0°  PINC=0x147AE147  POFF=0x00000000
CD: 5.000 MHz  0.0°  PINC=0x147AE147  POFF=0x00000000

✓ DAC2 DDS configured (62.5 MHz clock):
  AB: 5.0 MHz @ 0.0°
  CD: 5.0 MHz @ 0.0°


In [8]:
zynq.adc_write_register(0x005, 0x03)   # chan1 only
# zynq.adc_write_register(0x015, 0x00)   # 3.72mA output drive current
zynq.adc_write_register(0x017, 0xFF)   # no clock delay
# zynq.adc_write_register(0x018, 0x1F)   # no clock delay
zynq.adc_write_register(0x0FF, 0x01)   # latch


True

In [7]:
## Add FPGACLK in order to start all LVDS data and clocks
# DACCLK1 DACCLK2 1 GHz, OSTR effective 15.625 MHz, FPGA_CLK 250 MHz, ADC_CLK dependes on config word
CDCE_CONFIG_wADC125 = '8340032083400321EB040302811C0303EB84031410000BE504BE09F6BD0037F7'
CDCE_CONFIG_wADC250 = '8340032083400321EB840302811C0303EB84031410000BE504BE09F6BD0037F7'
zynq.configure_clock(CDCE_CONFIG_wADC250)
print('CDCE configured with FPGACLK and ADCCLK')

CDCE_CONFIG_wADC250 ='8340032083400321EB840302811C0303EB84031410000BE504BE09F6BD0037F7'



CDCE configured with FPGACLK and ADCCLK


---
## ADC diagnostics

In [9]:
# Read back all key ADC registers
zynq.adc_read_all_registers()


[ADC] Reading key registers:
  0x001: 0x82
  0x002: 0x00
  0x005: 0x03
  0x008: 0x00
  0x009: 0x01
  0x00B: 0x00
  0x00D: 0x00
  0x010: 0x00
  0x014: 0x05
  0x015: 0x00
  0x016: 0x00
  0x017: 0xFF
  0x018: 0x00
  0x03A: 0x00


{1: 130,
 2: 0,
 5: 3,
 8: 0,
 9: 1,
 11: 0,
 13: 0,
 16: 0,
 20: 5,
 21: 0,
 22: 0,
 23: 255,
 24: 0,
 58: 0}

In [ ]:
# ADC test pattern — checkerboard (0x555 / 0xAAA alternating)
# Useful to verify LVDS lanes are wired and captured correctly.
# Expected output on FPGA capture: 0x555, 0xAAA, 0x555, 0xAAA ...
zynq.adc_write_register(0x00D, 0x04)   # test mode = checkerboard
zynq.adc_write_register(0x0FF, 0x01)   # latch
print('ADC test mode: checkerboard')

In [13]:
# ADC test pattern — ramp
# Each sample increments by 1. Verifies sample ordering and bit integrity.
zynq.adc_write_register(0x00D, 0x0F)   # test mode = ramp
zynq.adc_write_register(0x0FF, 0x01)   # latch
print('ADC test mode: ramp')

ADC test mode: ramp


In [ ]:
# ADC test pattern — midscale short (all zeros)
zynq.adc_write_register(0x00D, 0x01)   # test mode = midscale short
zynq.adc_write_register(0x0FF, 0x01)   # latch
print('ADC test mode: midscale short (output = 0x000)')

In [ ]:
# ADC test pattern — positive full scale
zynq.adc_write_register(0x00D, 0x02)   # test mode = positive FS
zynq.adc_write_register(0x0FF, 0x01)   # latch
print('ADC test mode: positive full scale (output = 0x7FF)')

In [ ]:
# ADC test pattern — negative full scale
zynq.adc_write_register(0x00D, 0x03)   # test mode = negative FS
zynq.adc_write_register(0x0FF, 0x01)   # latch
print('ADC test mode: negative full scale (output = 0x800)')

In [ ]:
# Disable test mode — return to normal ADC sampling
zynq.adc_write_register(0x00D, 0x00)   # test mode = off
zynq.adc_write_register(0x0FF, 0x01)   # latch
print('ADC test mode: OFF (normal operation)')

---
## Clock / power diagnostics

In [ ]:
# Read back all CDCE registers
zynq.cdce.read_all()

In [76]:
# Read back all DAC registers (both DACs)
# zynq.read_all_dac_registers(dac_num=1)
zynq.read_all_dac_registers(dac_num=2)


[DAC2] Reading all registers:
  0x00: 0x089B
  0x01: 0x050E
  0x02: 0xF052
  0x03: 0xF001
  0x04: 0xDA4C
  0x05: 0x3978
  0x06: 0x3300
  0x07: 0xD8FF
  0x08: 0x0000
  0x09: 0x8000
  0x0A: 0x0000
  0x0B: 0x0000
  0x0C: 0x0400
  0x0D: 0x0400
  0x0E: 0x0400
  0x0F: 0x0400
  0x10: 0x0000
  0x11: 0x0000
  0x12: 0x0000
  0x13: 0x0000
  0x14: 0x0000
  0x15: 0x0000
  0x16: 0x0000
  0x17: 0x0000
  0x18: 0x0466
  0x19: 0x0434
  0x1A: 0xFC00
  0x1B: 0x0800
  0x1C: 0x0000
  0x1D: 0x0000
  0x1E: 0x8888
  0x1F: 0x2228
  0x20: 0x0000
  0x21: 0x0000
  0x22: 0x1B1B
  0x23: 0xFFFF
  0x24: 0x1C00
  0x25: 0x7A7A
  0x26: 0xB6B6
  0x27: 0xEAEA
  0x28: 0x4545
  0x29: 0x1A1A
  0x2A: 0x1616
  0x2B: 0xAAAA
  0x2C: 0xC6C6
  0x2D: 0x0004
  0x2E: 0x0000
  0x2F: 0x0000
  0x30: 0x0000
  0x7F: 0x540C


{0: 2203,
 1: 1294,
 2: 61522,
 3: 61441,
 4: 55884,
 5: 14712,
 6: 13056,
 7: 55551,
 8: 0,
 9: 32768,
 10: 0,
 11: 0,
 12: 1024,
 13: 1024,
 14: 1024,
 15: 1024,
 16: 0,
 17: 0,
 18: 0,
 19: 0,
 20: 0,
 21: 0,
 22: 0,
 23: 0,
 24: 1126,
 25: 1076,
 26: 64512,
 27: 2048,
 28: 0,
 29: 0,
 30: 34952,
 31: 8744,
 32: 0,
 33: 0,
 34: 6939,
 35: 65535,
 36: 7168,
 37: 31354,
 38: 46774,
 39: 60138,
 40: 17733,
 41: 6682,
 42: 5654,
 43: 43690,
 44: 50886,
 45: 4,
 46: 0,
 47: 0,
 48: 0,
 127: 21516}